# Missing-teeth classifier (gap vs tooth)

The zero-training derivation (interior FDI gaps from the 32-class tooth
detector) scores **F1 0.742** on the held-out 94-image split: precision 0.699
(some "gaps" are really teeth the detector missed) and recall 0.792 (some real
gaps are filled by a hallucinated detection). Both errors are about *tooth
presence at a slot*, so a single binary classifier can attack both.

Design — one ResNet18, used symmetrically:

* **Localize** every interior slot (FDI tooth 2-7) the same way training and
  inference: use the detector's box if it fired, otherwise interpolate between
  the nearest detected neighbours.
* **Label** by DENTEX enumeration GT: slot present -> `tooth` (0), slot absent
  -> `gap` (1).
* At inference: reject a detected tooth if the crop looks like a gap (recall),
  and keep a candidate gap only if the crop looks like a gap (precision).

Trained on 540 enumeration images, evaluated on the held-out 94 (the tooth
detector's own split). Compare against the F1 0.742 derivation.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

WHEELS = next(Path("/kaggle/input").rglob("yolov8s.pt")).parent
os.environ["YOLO_OFFLINE"] = "True"
try:
    import ultralytics
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                    "--find-links", str(WHEELS), "ultralytics", "ultralytics-thop"],
                   check=True)
    import ultralytics

import torch
import torchvision
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| torchvision", torchvision.__version__, "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Verify the Kaggle account is phone-verified.")
print("gpu:", torch.cuda.get_device_properties(0).name)


In [ ]:
import csv
import json
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
from PIL import Image
from ultralytics import YOLO

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
SIZE, PAD = 224, 0.10
INTERIOR = range(2, 8)   # FDI 2..7 (third molar excluded)

QE = next(INPUT.rglob("train_quadrant_enumeration.json"))
IMG = QE.parent / "xrays"
qe = json.load(open(QE))
print("images", len(qe["images"]), "boxes", len(qe["annotations"]), "|", IMG)

imgs = {i["id"]: i for i in qe["images"]}
gt = defaultdict(set)
labels = defaultdict(list)
for a in qe["annotations"]:
    name = imgs[a["image_id"]]["file_name"]
    gt[name].add((a["category_id_1"] + 1, a["category_id_2"] + 1))
    labels[name].append(a)

# exact tooth-detector split (540/94, seed 42)
groups = defaultdict(list)
for i in qe["images"]:
    n = len(labels[i["file_name"]])
    groups["sparse" if n < 20 else ("dense" if n > 30 else "normal")].append(i["file_name"])
rng = random.Random(42)
train_names, val_names = [], []
for key in sorted(groups, key=str):
    m = sorted(groups[key]); rng.shuffle(m)
    k = max(1, round(len(m) * 0.15)) if len(m) > 1 else 0
    val_names.extend(m[:k]); train_names.extend(m[k:])
assert len(train_names) == 540 and len(val_names) == 94
print(f"split: train {len(train_names)} | val {len(val_names)}")

TOOTH = next(INPUT.rglob("tooth_best.pt"))
tooth = YOLO(str(TOOTH))


def iou(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    if inter <= 0:
        return 0.0
    return inter / ((a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)


def detect(name, conf=0.01):
    res = tooth.predict(str(IMG / name), imgsz=1024, conf=conf, verbose=False)[0]
    found = {}
    if res.boxes is None:
        return found, (0, 0)
    H, W = res.orig_shape
    raw = [(int(b.cls[0].item()), float(b.conf[0].item()),
            [float(v) for v in b.xyxy[0].cpu().numpy()]) for b in res.boxes]
    raw.sort(key=lambda r: r[1], reverse=True)
    keep = []
    for cls, c, box in raw:
        if all(iou(box, k[2]) < 0.6 for k in keep):
            keep.append((cls, c, box))
    for cls, c, box in keep:
        q, t = cls // 8 + 1, cls % 8 + 1
        if (q, t) not in found or c > found[(q, t)][1]:
            found[(q, t)] = (box, c)
    return found, (W, H)


In [ ]:
# ---- cache one detector pass over all 634 images ----
CACHE = WORK / "detect_cache.json"
if CACHE.exists():
    cache = json.load(open(CACHE))
    cache = {k: {tuple(int(x) for x in kk.split("_")): (v[0], v[1]) for kk, v in d.items()}
             for k, d in cache.items()}
else:
    cache = {}
    for i, n in enumerate(train_names + val_names, 1):
        found, _ = detect(n)
        cache[n] = {f"{q}_{t}": [box, c] for (q, t), (box, c) in found.items()}
        if i % 100 == 0:
            print(f"  detect {i}/{len(train_names)+len(val_names)}", flush=True)
    json.dump(cache, open(CACHE, "w"))
    cache = {k: {tuple(int(x) for x in kk.split("_")): (v[0], v[1]) for kk, v in d.items()}
             for k, d in cache.items()}
print("cached images:", len(cache))


In [ ]:
# ---- build crops, localized exactly as at inference ----
CROPS = WORK / "mt_crops"


def interp_box(pt, t):
    lo = [x for x in pt if x < t]
    hi = [x for x in pt if x > t]
    if not lo or not hi:
        return None
    L, H = max(lo), min(hi)
    bl, bh = pt[L][0], pt[H][0]
    a = (t - L) / (H - L)
    x = bl[0]*(1-a) + bh[0]*a
    y = bl[1]*(1-a) + bh[1]*a
    w = (bl[2]-bl[0] + bh[2]-bh[0]) / 2
    h = (bl[3]-bl[1] + bh[3]-bh[1]) / 2
    return [x - w/2, y - h/2, x + w/2, y + h/2]


def crop_pil(im, box):
    x1, y1, x2, y2 = box
    w, h = x2-x1, y2-y1
    x1, x2 = x1 - PAD*w, x2 + PAD*w
    y1, y2 = y1 - PAD*h, y2 + PAD*h
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(im.width, x2), min(im.height, y2)
    if x2-x1 < 8 or y2-y1 < 8:
        return None
    return im.crop((x1, y1, x2, y2)).resize((SIZE, SIZE), Image.BILINEAR)


rows = []
for split, names in (("train", train_names), ("val", val_names)):
    for n in names:
        pt = cache[n]
        im = Image.open(IMG / n).convert("RGB")
        for q in (1, 2, 3, 4):
            present = {t: pt[(q, t)] for t in INTERIOR if (q, t) in pt}
            for t in INTERIOR:
                box = present[t][0] if t in present else interp_box(pt, t)
                if box is None:
                    continue
                label = 1 if (q, t) not in gt[n] else 0   # 1 = gap
                crop = crop_pil(im, box)
                if crop is None:
                    continue
                dst_ = CROPS / split / ("gap" if label else "tooth") / f"{n[:-4]}_{q}{t}.jpg"
                dst_.parent.mkdir(parents=True, exist_ok=True)
                crop.save(dst_, quality=92)
                rows.append({"path": str(dst_), "label": label, "split": split,
                             "image": n, "slot": f"{q}{t}"})

# balance: cap tooth crops at 2x gap crops per split
kept = []
for split in ("train", "val"):
    r = [x for x in rows if x["split"] == split]
    gaps = [x for x in r if x["label"] == 1]
    teeth = [x for x in r if x["label"] == 0]
    random.Random(42).shuffle(teeth)
    keep_n = len(teeth) if split == "val" else min(len(teeth), 2 * len(gaps))
    kept.extend(gaps + teeth[:keep_n])
with open(CROPS / "manifest.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["path", "label", "split", "image", "slot"])
    w.writeheader(); w.writerows(kept)
c = defaultdict(int)
for x in kept:
    c[(x["split"], x["label"])] += 1
print("crops:", dict(c), "| total", len(kept))


In [ ]:
# ---- train ResNet18 ----
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
EPOCHS, BATCH = 6, 64


def seed_all(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


class C(Dataset):
    def __init__(self, rows, train):
        self.rows = rows
        if train:
            self.tf = transforms.Compose([
                transforms.RandomHorizontalFlip(),
                transforms.RandomAffine(5, translate=(0.05, 0.05), scale=(0.9, 1.1)),
                transforms.ColorJitter(0.2, 0.2), transforms.ToTensor(),
                transforms.Normalize(MEAN, STD)])
        else:
            self.tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        return self.tf(Image.open(r["path"]).convert("RGB")), int(r["label"])


def auc(y, p):
    y, p = np.asarray(y), np.asarray(p)
    o = np.argsort(p); y = y[o]
    n1 = y.sum(); n0 = len(y) - n1
    if n0 == 0 or n1 == 0:
        return float("nan")
    r = np.arange(1, len(y) + 1)
    return float((r[y == 1].sum() - n1*(n1+1)/2) / (n0*n1))


rows = list(csv.DictReader(open(CROPS / "manifest.csv")))
for r in rows:
    r["label"] = int(r["label"])
train = [r for r in rows if r["split"] == "train"]
val = [r for r in rows if r["split"] == "val"]
n1 = sum(r["label"] for r in train); n0 = len(train) - n1
print(f"train {len(train)} (gap {n1}) | val {len(val)} (gap {sum(r['label'] for r in val)})")

seed_all()
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).cuda()
model.fc = nn.Linear(model.fc.in_features, 2).cuda()
w = torch.tensor([len(train)/(2*n0), len(train)/(2*n1)], dtype=torch.float32).cuda()
crit = nn.CrossEntropyLoss(weight=w)
opt = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
tr = DataLoader(C(train, True), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
va = DataLoader(C(val, False), batch_size=128, shuffle=False, num_workers=2)

best = -1
for ep in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); tot = seen = 0
    for i, (x, y) in enumerate(tr, 1):
        x, y = x.cuda(), y.cuda()
        opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
        tot += loss.item()*len(y); seen += len(y)
        if i % 50 == 0:
            print(f"  ep {ep} {i}/{len(tr)} loss {tot/seen:.4f} {(time.time()-t0)/i:.2f}s/it", flush=True)
    sch.step()
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y in va:
            ps.extend(torch.softmax(model(x.cuda()), 1)[:, 1].cpu().tolist())
            ys.extend(y.tolist())
    a = auc(ys, ps)
    print(f"epoch {ep}: val gap AUC {a:.4f}", flush=True)
    if a > best:
        best = a
        torch.save({"state_dict": model.state_dict(), "arch": "resnet18", "size": SIZE,
                    "mean": MEAN, "std": STD, "val_auc": a, "epoch": ep},
                   WORK / "missing_teeth.pt")
print("best val AUC", best)


In [ ]:
# ---- evaluate: derivation vs classifier on the held-out 94 ----
import torch
from torchvision import transforms

ckpt = torch.load(WORK / "missing_teeth.pt", map_location="cpu")
model.load_state_dict(ckpt["state_dict"]); model.eval()
TF = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


@torch.no_grad()
def gap_prob(crop):
    x = TF(crop).unsqueeze(0).cuda()
    return float(torch.softmax(model(x), 1)[0, 1])


def gt_missing(n):
    out = set()
    for q in (1, 2, 3, 4):
        teeth = sorted(t for (qq, t) in gt[n] if qq == q)
        for t in INTERIOR:
            if t not in teeth and any(x < t for x in teeth) and any(x > t for x in teeth):
                out.add((q, t))
    return out


def missing_derived(n):
    pt = {t: cache[n][(q, t)] for q in (1, 2, 3, 4) for t in INTERIOR if (q, t) in cache[n]}
    out = set()
    for q in (1, 2, 3, 4):
        te = {t: cache[n][(q, t)] for t in INTERIOR if (q, t) in cache[n]}
        for t in INTERIOR:
            if t not in te and any(x < t for x in te) and any(x > t for x in te):
                out.add((q, t))
    return out


def missing_clf(n, t_tooth, t_gap):
    im = Image.open(IMG / n).convert("RGB")
    present = {}
    for q in (1, 2, 3, 4):
        te = {t: cache[n][(q, t)] for t in INTERIOR if (q, t) in cache[n]}
        # 1) validate detected teeth: drop if crop looks like a gap
        for t in list(te):
            crop = crop_pil(im, te[t][0])
            if crop is not None and gap_prob(crop) >= t_tooth:
                del te[t]
        # 2) candidate gaps: absent with detected neighbours
        for t in INTERIOR:
            if t in te:
                continue
            box = interp_box({**{tt: te[tt] for tt in te}}, t)
            if box is None:
                continue
            crop = crop_pil(im, box)
            if crop is not None and gap_prob(crop) >= t_gap:
                present.setdefault(q, set()).add(t)
    return {(q, t) for q, ts in present.items() for t in ts}


def score(pred_fn):
    tp = fp = fn = 0
    for n in val_names:
        g = gt_missing(n); p = pred_fn(n)
        tp += len(g & p); fp += len(p - g); fn += len(g - p)
    pr = tp/max(1, tp+fp); rc = tp/max(1, tp+fn)
    return tp, fp, fn, pr, rc, 2*pr*rc/max(1e-9, pr+rc)


tp, fp, fn, pr, rc, f1 = score(missing_derived)
print(f"derivation     : P {pr:.3f} R {rc:.3f} F1 {f1:.3f} (tp {tp} fp {fp} fn {fn})")

best = None
for t_gap in (0.5, 0.6, 0.7, 0.8, 0.9):
    for t_tooth in (1.1, 0.9, 0.8, 0.7, 0.6):
        r = score(lambda n: missing_clf(n, t_tooth, t_gap))
        if best is None or r[5] > best[0][5]:
            best = (r, t_tooth, t_gap)
        print(f"  clf t_tooth {t_tooth:.1f} t_gap {t_gap:.1f}: "
              f"P {r[3]:.3f} R {r[4]:.3f} F1 {r[5]:.3f} (tp {r[0]} fp {r[1]} fn {r[2]})")

(r, t_tooth, t_gap) = best
print(f"\nBEST clf: t_tooth {t_tooth} t_gap {t_gap} -> "
      f"P {r[3]:.3f} R {r[4]:.3f} F1 {r[5]:.3f}")
summary = {"derivation": {"f1": f1, "precision": pr, "recall": rc},
           "classifier": {"f1": r[5], "precision": r[3], "recall": r[4],
                          "t_tooth": t_tooth, "t_gap": t_gap},
           "val_auc": ckpt["val_auc"], "train_images": 540, "val_images": 94}
(WORK / "missing_teeth_metrics.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))


In [ ]:
import shutil

shutil.rmtree(CROPS, ignore_errors=True)
print("output:", sorted(p.name for p in WORK.iterdir()))
